# Lesson 4.6: Time Series in Pandas

**Analyst:** John Ko
**Date:** 11/08/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Produce the cohort churn breakdown Hannah asked for
this morning, plus supporting monthly-signup trend and YoY comparison.
Deliverable for end-of-week.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv(
    'finflow-enriched-200k.csv',
    parse_dates=['signup_date', 'last_login_date', 'churn_date']
)

print(f"Loaded {len(df):,} rows")
print(f"Signup date range: {df['signup_date'].min()} to {df['signup_date'].max()}")
print(f"Columns: {df.shape[1]}")

Loaded 200,000 rows
Signup date range: 2024-01-02 00:00:00 to 2026-01-01 00:00:00
Columns: 22


## Building monthly cohorts

Each customer's signup_date becomes a signup_cohort (their signup month).
This is the cohort identifier we'll group by.


In [2]:
df['signup_cohort'] = df['signup_date'].dt.to_period('M')

# Inspect
print(df[['customer_id', 'signup_date', 'signup_cohort']].head())
print()
print(f"Type of signup_cohort: {df['signup_cohort'].dtype}")
print(f"Number of distinct cohorts: {df['signup_cohort'].nunique()}")

  customer_id signup_date signup_cohort
0    FF-00001  2025-10-17       2025-10
1    FF-00002  2025-11-24       2025-11
2    FF-00003  2024-08-12       2024-08
3    FF-00004  2025-03-06       2025-03
4    FF-00005  2025-12-16       2025-12

Type of signup_cohort: period[M]
Number of distinct cohorts: 25


In [3]:
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

cohort_churn = df.groupby('signup_cohort').agg(
    customer_count=('customer_id', 'count'),
    churn_rate_pct=('active', churn_rate_pct),
    avg_monthly_fee=('monthly_fee', 'mean')
).round(2)

print(cohort_churn.head(10))
print("--- (rows omitted) ---")
print(cohort_churn.tail(5))

               customer_count  churn_rate_pct  avg_monthly_fee
signup_cohort                                                 
2024-01                  8303           20.41            33.89
2024-02                  7870           20.48            35.01
2024-03                  8542           20.44            35.64
2024-04                  8349           19.60            34.37
2024-05                  8601           20.44            34.42
2024-06                  8092           20.58            35.32
2024-07                  8494           20.19            34.48
2024-08                  8516           19.39            34.58
2024-09                  8069           19.31            33.64
2024-10                  8364           20.31            35.09
--- (rows omitted) ---
               customer_count  churn_rate_pct  avg_monthly_fee
signup_cohort                                                 
2025-09                  8198           19.90            34.79
2025-10                  8724   

**Observation:** Cohort churn is flat, 19.16% to 20.12%, with no relationship
to cohort age. Reporting this as "churn is stable across cohorts" would be
technically true and misleading, because current churn rate is the wrong
measure: it mixes cohorts that have had two years to churn with ones that
have had ten days. The honest answer to Hannah is that this cut does not
separate the cohorts, and the measure that would is churn at equal age,
separate the cohorts, and the measure that would is churn at equal age,
e.g. churn rate at six months since signup. Flag that for Diego. That's a
follow-up, not today's deliverable.


In [4]:
cohort_churn.to_csv('cohort-churn-by-signup-month.csv')
print("Saved cohort-churn-by-signup-month.csv")

Saved cohort-churn-by-signup-month.csv


In [5]:
# Set signup_date as index for resampling
df_indexed = df.set_index('signup_date')

# Monthly signup counts
monthly_signups = df_indexed.resample('ME').size()
print(monthly_signups.head(10))
print(f"\nTotal months: {len(monthly_signups)}")

signup_date
2024-01-31    8303
2024-02-29    7870
2024-03-31    8542
2024-04-30    8349
2024-05-31    8601
2024-06-30    8092
2024-07-31    8494
2024-08-31    8516
2024-09-30    8069
2024-10-31    8364
Freq: ME, dtype: int64

Total months: 25


In [6]:
# Three-month rolling average of monthly signups
monthly_signups_smoothed = monthly_signups.rolling(window=3).mean()

# Compare side by side
comparison = pd.DataFrame({
    'raw': monthly_signups,
    'smooth_3mo': monthly_signups_smoothed.round(0)
})
print(comparison.head(8))


              raw  smooth_3mo
signup_date                  
2024-01-31   8303         NaN
2024-02-29   7870         NaN
2024-03-31   8542     8238.00
2024-04-30   8349     8254.00
2024-05-31   8601     8497.00
2024-06-30   8092     8347.00
2024-07-31   8494     8396.00
2024-08-31   8516     8367.00


In [7]:
# Month-over-month percent change in signups
mom_change = monthly_signups.pct_change() * 100

print(mom_change.head(8).round(2))

signup_date
2024-01-31     NaN
2024-02-29   -5.21
2024-03-31    8.54
2024-04-30   -2.26
2024-05-31    3.02
2024-06-30   -5.92
2024-07-31    4.97
2024-08-31    0.26
Freq: ME, dtype: float64


In [8]:
# YoY change in signups
monthly_yoy_prev = monthly_signups.shift(12)
yoy_change = monthly_signups - monthly_yoy_prev
yoy_pct = (monthly_signups / monthly_yoy_prev - 1) * 100

yoy_comparison = pd.DataFrame({
    'current': monthly_signups,
    'prev_year': monthly_yoy_prev,
    'change': yoy_change,
    'pct_change': yoy_pct.round(2)
})

# Show rows where YoY is computable (i.e., second year onward)
print(yoy_comparison.dropna().head(8))

             current  prev_year  change  pct_change
signup_date                                        
2025-01-31      8458    8303.00  155.00        1.87
2025-02-28      7539    7870.00 -331.00       -4.21
2025-03-31      8424    8542.00 -118.00       -1.38
2025-04-30      8143    8349.00 -206.00       -2.47
2025-05-31      8476    8601.00 -125.00       -1.45
2025-06-30      8231    8092.00  139.00        1.72
2025-07-31      8459    8494.00  -35.00       -0.41
2025-08-31      8434    8516.00  -82.00       -0.96


In [9]:
ttm_signups = monthly_signups.rolling(window=12).sum()
print(ttm_signups.dropna().head(5))

signup_date
2024-12-31    99973.00
2025-01-31   100128.00
2025-02-28    99797.00
2025-03-31    99679.00
2025-04-30    99473.00
Freq: ME, dtype: float64


**AI Use Note:** Used Claude to confirm the TTM rolling-sum pattern.
Output matched the obvious rolling(window=12).sum() idiom. Verified.
Tier 1 disclosure.